# Pretrained ResNet
Выберите конфиг, задайте уникальный EXP_ID и выполните ячейки сверху вниз.
`head_only`: новая голова; `partial`: выбранные блоки и голова; `full`: вся сеть.
В partial/full первые три эпохи обучается только голова. Графики, метрики и артефакты создаёт общий pipeline.
`auto` выбирает CUDA при наличии, иначе CPU. CPU поддерживается, но обучение может быть заметно дольше.
Нужен torchvision, совместимый с torch. При первом запуске веса скачиваются, если их нет в кеше; на Kaggle для этого нужен Internet.
Test inference по умолчанию выключен. Подробности: `docs/transfer_learning.md`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "experiment.py").exists():
    raise RuntimeError("Откройте ноутбук из корня проекта или notebooks/")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.experiment import load_config, run_experiment
from src.utils import get_device


In [ ]:
# Следующий новый запуск: ResNet101 с лучшим текущим рецептом.
CONFIG = "configs/resnet101_partial_domain_shift_stronger.json"
EXP_ID = "resnet101_partial_domain_shift_stronger_001"  # Уникальный ID
DEVICE = "auto"

config = load_config(CONFIG, project_root=PROJECT_ROOT)
config["training"]["device"] = DEVICE
# Если batch 32 не помещается в GPU, уменьшите batch_size, сохраняя image_size=256.
# config["data"]["batch_size"] = 16
print(f"Конфигурация: {CONFIG}; устройство: {get_device(DEVICE)}")


In [ ]:
result = run_experiment(config, exp_id=EXP_ID, project_root=PROJECT_ROOT)


## После обучения

Сохраните artifacts/models, artifacts/logs, artifacts/predictions и новую строку experiments.csv. Новый pipeline сохраняет также val_probabilities.npz: IDs, порядок классов, вероятности и true_labels.
Для сравнения TTA/ансамбля откройте 05_error_analysis.ipynb и добавьте новый best.pt в CHECKPOINTS, задав новый INFERENCE_RUN_ID. Test не используется при обучении и выборе.
